<a href="https://colab.research.google.com/github/adelinda02/Gen-AI/blob/main/Text_Classification_with_NLTK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import nltk
import sklearn
import pandas as pd
import numpy as np

In [2]:
print('Python: {}'.format(sys.version))
print('NLTK: {}'.format(nltk.__version__))
print('Scikit-learn: {}'.format(sklearn.__version__))
print('Pandas: {}'.format(pd.__version__))
print('NumPy: {}'.format(np.__version__))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NLTK: 3.9.1
Scikit-learn: 1.6.1
Pandas: 2.2.2
NumPy: 2.0.2


In [3]:
# Load the dataset of SMS messages
import os
import requests
import zipfile
import io

file_name = 'SMSSpamCollection'
zip_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'

if not os.path.exists(file_name):
    print(f"Downloading {file_name}...")
    try:
        response = requests.get(zip_url)
        response.raise_for_status() # Raise an exception for HTTP errors
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extract(file_name)
        print(f"Successfully downloaded and extracted {file_name}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the dataset: {e}")
        print("Please ensure you have an active internet connection or manually upload the file.")
    except KeyError:
        print(f"Error: '{file_name}' not found inside the zip file '{zip_url}'.")
        print("Please check the contents of the zip or manually upload the correct file.")
    except Exception as e:
        print(f"An unexpected error occurred during download/extraction: {e}")
        print("Please consider manually uploading the file.")

sms = pd.read_table(file_name, header=None, encoding='utf-8')
sms.head()

Successfully downloaded and extracted SMSSpamCollection


,0,1
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
sms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5572 non-null   object
 1   1       5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [5]:
# Check class distribution
sms[0].value_counts()

,count
0,
ham,4825
spam,747


In [6]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
label = enc.fit_transform(sms[0])
print(label[:10])
print(sms[0][:10])

[0 0 1 0 0 1 0 0 1 1]
0     ham
1     ham
2    spam
3     ham
4     ham
5    spam
6     ham
7     ham
8    spam
9    spam
Name: 0, dtype: object


In [7]:
text = sms[1]
text[:10]

,1
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...
8,WINNER!! As a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...


In [8]:
# Use regular expression

# Replace email addresses with 'email'
processed = text.str.replace(r'^.+@[^\.].*\.[a-z]{2,}$', 'emailaddress')

# Replace URLs with 'webaddress'
processed = processed.str.replace(r'^http\://[a-zA-Z0-9\-\.]+\.[a-zA-Z]{2,3}(/\S*)?$', 'webaddress')

# Replace money symbols with 'moneysymb' (£ can by typed with ALT key + 156)
processed = processed.str.replace(r'£|\$', 'moneysymb')

# Replace 10 digit phone numbers (formats include paranthesis, spaces, no spaces, dashes) with 'phonenumber'
processed = processed.str.replace(r'^\(?[\d]{3}\)?[\s-]?[\d]{3}[\s-]?[\d]{4}$', 'phonenumbr')

# Replace numbers with 'numbr'
processed = processed.str.replace(r'\d+(\.\d+)?', 'numbr')

In [9]:
processed = processed.str.replace(r'[^\w\d\s]', ' ')

# Replace whitespace between terms with a single space
processed = processed.str.replace(r'\s+', ' ')

# Remove leading and trailing whitespace
processed = processed.str.replace(r'^\s+|\s+?$', '')

In [10]:
processed = processed.str.lower()
processed

,1
0,"go until jurong point, crazy.. available only ..."
1,ok lar... joking wif u oni...
2,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor... u c already then say...
4,"nah i don't think he goes to usf, he lives aro..."
...,...
5567,this is the 2nd time we have tried 2 contact u...
5568,will ü b going to esplanade fr home?
5569,"pity, * was in mood for that. so...any other s..."
5570,the guy did some bitching but i acted like i'd...


In [11]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

processed = processed.apply(lambda x: ' '.join(term for term in x.split() if term not in stop_words))

In [12]:
ps = nltk.PorterStemmer()

processed = processed.apply(lambda x: ' '.join(ps.stem(term) for term in x.split()))

In [13]:
processed

,1
0,"go jurong point, crazy.. avail bugi n great wo..."
1,ok lar... joke wif u oni...
2,free entri 2 wkli comp win fa cup final tkt 21...
3,u dun say earli hor... u c alreadi say...
4,"nah think goe usf, live around though"
...,...
5567,2nd time tri 2 contact u. u £750 pound prize. ...
5568,ü b go esplanad fr home?
5569,"pity, * mood that. so...ani suggestions?"
5570,guy bitch act like interest buy someth els nex...


In [14]:
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt_tab', quiet=True)

all_words = []

for message in processed:
    words = word_tokenize(message)
    for w in words:
        all_words.append(w)

all_words = nltk.FreqDist(all_words)

# Print the result
print('Number of words: {}'.format(len(all_words)))
print('Most common words: {}'.format(all_words.most_common(15)))

Number of words: 8920
Most common words: [('.', 4759), (',', 1939), ('?', 1550), ('!', 1397), ('...', 1146), ('u', 1138), ('&', 922), (';', 768), (':', 722), ('..', 697), ('call', 644), (')', 499), ('2', 478), ('go', 446), ('get', 446)]


In [15]:
# use the 1500 most common words as features
word_features = [x[0] for x in all_words.most_common(1500)]

In [16]:
def find_features(message):
    words = word_tokenize(message)
    features = {}
    for word in word_features:
        features[word] = (word in words)

    return features

In [17]:
features = find_features(processed[0])
for key, value in features.items():
    if value == True:
        print(key)


,
...
..
go
got
n
great
wat
e
world
point
avail
la
cine


In [18]:
list(features.items())[:10]

[('.', False),
 (',', True),
 ('?', False),
 ('!', False),
 ('...', True),
 ('u', False),
 ('&', False),
 (';', False),
 (':', False),
 ('..', True)]

In [19]:
messages = list(zip(processed, label))

np.random.seed(1)
np.random.shuffle(messages)

# Call find_features function for each SMS message
feature_set = [(find_features(text), label) for (text, label) in messages]
from sklearn.model_selection import train_test_split

training, test = train_test_split(feature_set, test_size=0.25, random_state=1)
print(len(training))
print(len(test))

4179
1393


In [20]:
from nltk.classify.scikitlearn import SklearnClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

names = ['K Nearest Neighbors', 'Decision Tree', 'Random Forest', 'Logistic Regression', 'SGD Classifier',
         'Naive Bayes', 'Support Vector Classifier']

classifiers = [
    KNeighborsClassifier(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    LogisticRegression(),
    SGDClassifier(max_iter=100),
    MultinomialNB(),
    SVC(kernel='linear')
]

models = zip(names, classifiers)

for name, model in models:
    nltk_model = SklearnClassifier(model)
    nltk_model.train(training)
    accuracy = nltk.classify.accuracy(nltk_model, test)
    print("{} model Accuracy: {}".format(name, accuracy))

K Nearest Neighbors model Accuracy: 0.9152907394113424
Decision Tree model Accuracy: 0.9569274946159368
Random Forest model Accuracy: 0.9791816223977028
Logistic Regression model Accuracy: 0.9806173725771715
SGD Classifier model Accuracy: 0.9770279971284996
Naive Bayes model Accuracy: 0.9877961234745154
Support Vector Classifier model Accuracy: 0.9827709978463748


In [21]:
from sklearn.ensemble import VotingClassifier

# Since VotingClassifier can accept list type of models
models = list(zip(names, classifiers))

nltk_ensemble = SklearnClassifier(VotingClassifier(estimators=models, voting='hard', n_jobs=-1))
nltk_ensemble.train(training)
accuracy = nltk.classify.accuracy(nltk_ensemble, test)
print("Voting Classifier model Accuracy: {}".format(accuracy))

Voting Classifier model Accuracy: 0.9827709978463748


In [22]:
text_features, labels = zip(*test)
prediction = nltk_ensemble.classify_many(text_features)
print(classification_report(labels, prediction))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1199
           1       0.99      0.88      0.93       194

    accuracy                           0.98      1393
   macro avg       0.99      0.94      0.96      1393
weighted avg       0.98      0.98      0.98      1393



In [23]:
pd.DataFrame( confusion_matrix(labels, prediction),
             index=[['actual', 'actual'], ['ham', 'spam']],
             columns = [['predicted', 'predicted'], ['ham', 'spam']])

predicted     
                  ham spam
actual ham       1198    1
       spam        23  171